# 03/01 — Pseudobulk per (mouse, region)

Collapse spot-level counts to one row per (sample x region). Everything downstream (DGE, attenuation slope, mixed model) consumes this table, so the unit of replication is the *mouse* rather than the spot.

Outputs:
* `results/tables/attenuation/pseudobulk_counts.tsv`
* `results/tables/attenuation/pseudobulk_meta.tsv`

In [11]:
from __future__ import annotations
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# regional-annotation h5ad lives on the processing volume
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'   # raw integer counts live here, not in .X

TBL = ROOT / 'results' / 'tables' / 'attenuation'
TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'
FIG.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY    = 'sample_id'           # change to 'library_id' if obs uses that
REGION_KEY    = 're_annotation_regions'   # adjust to your obs column for regions
TREATMENT_KEY = 'treatment'
print('h5ad        :', H5AD)
print('count layer :', COUNT_LAYER)


h5ad        : /Volumes/processing2/ST_BRICHOS/data/ST_BRICHOS_region_subcluster.h5ad
count layer : counts


In [12]:
from utils.attenuation import make_pseudobulk

adata = sc.read_h5ad(H5AD)
print(adata)
for k in (SAMPLE_KEY, REGION_KEY, TREATMENT_KEY):
    if k not in adata.obs.columns:
        cands = [c for c in adata.obs.columns if k.split('_')[0] in c]
        print(f'!! {k!r} missing — candidates: {cands}')


AnnData object with n_obs × n_vars = 25660 × 18751
    obs: 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres', 'sample', 'sample_id', 'n_genes', 'leiden', 'treatment', 'barcode', 'region_annotation', 'leiden_0.5', 'leiden_0.75', 'leiden_1', 'leiden_1.5', 'leiden_2', 'leiden_2.5', 'PIG_score', 'OLIG_score', 'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 're_annotation_regions'
    var: 'n_cells'
    uns: 'dendrogram_treatment', 'leiden', 'leiden_0.1_colors', 'leiden_0.2_colors', 'leiden_0.3_colors', 'leiden_0.5_colors', 'leiden_0.75_colors', 'leiden_1.5_colors', 'leiden_1_colors', 'leiden_2.5_colors', 'leiden_2_colors', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'region_annotation_colors', 'sample_id_colors', 'spatial', 'spatial_metadata_per_sample', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'


### Sum spot counts to (sample, region)

Raw counts live in `adata.layers['counts']`. `make_pseudobulk` reads that layer directly — no need to mutate `.X`.

In [13]:
counts, meta = make_pseudobulk(
    adata,
    sample_key=SAMPLE_KEY,
    region_key=REGION_KEY,
    treatment_key=TREATMENT_KEY,
    min_spots=20,
    layer=COUNT_LAYER,
)
# cast to integer counts (matches DESeq2/edgeR expectations downstream)
counts = counts.round().astype(int)
print(counts.shape, meta.shape)
meta.head()


(80, 18751) (80, 4)


,sample,region,treatment,n_spots
P24215_301__Caudoputamen,P24215_301,Caudoputamen,BRICHOS,533
P24215_301__Corpus callosum,P24215_301,Corpus callosum,BRICHOS,227
P24215_301__Hypothalamus,P24215_301,Hypothalamus,BRICHOS,339
P24215_301__Infragranular layers,P24215_301,Infragranular layers,BRICHOS,344
P24215_301__Meninges,P24215_301,Meninges,BRICHOS,122


### Sanity

In [14]:
print('mice per treatment:')
print(meta.groupby('treatment')['sample'].nunique())
print()
print('regions per treatment:')
print(meta.groupby(['treatment','region']).size().unstack(fill_value=0))


mice per treatment:
treatment
BRICHOS    3
PBS        6
WT         1
Name: sample, dtype: int64

regions per treatment:
region     Caudoputamen  Corpus callosum  Hippocampal formation  Hypothalamus  \
treatment                                                                       
BRICHOS               3                3                      1             3   
PBS                   6                6                      2             6   
WT                    1                1                      1             1   

region     Infragranular layers  Meninges  Olfactory areas  \
treatment                                                    
BRICHOS                       3         3                3   
PBS                           6         6                6   
WT                            1         1                1   

region     Supragranualar layers  Thalamus  Ventricular area  
treatment                                                     
BRICHOS                        3     

### Persist

In [15]:
counts.to_csv(TBL / 'pseudobulk_counts.tsv', sep='\t')
meta.to_csv  (TBL / 'pseudobulk_meta.tsv',   sep='\t')
print('wrote', TBL / 'pseudobulk_counts.tsv')
print('wrote', TBL / 'pseudobulk_meta.tsv')


wrote /Users/chrislangseth/work/karolinska_institutet/projects/ST-BRICHOS/results/tables/attenuation/pseudobulk_counts.tsv
wrote /Users/chrislangseth/work/karolinska_institutet/projects/ST-BRICHOS/results/tables/attenuation/pseudobulk_meta.tsv
